# JSON 04 - Writing JSON, and the traps

So far you only read. Now you **produce** JSON - which is what you do when you
save results, build an API response, or write a config.

Your data: **`../data/scores.json`**

```json
{
  "class": "L3-Info",
  "students": [
    { "name": "Amine", "scores": [14, 16, 11] },
    { "name": "Sara",  "scores": [18, 17, 19] },
    { "name": "Omar",  "scores": [9, 12, 10] },
    { "name": "Zoé",   "scores": [15, 15, 16] }
  ]
}
```

## Exercise 1 - Compute the averages

Load it, and print each student's average rounded to 2 decimals.

**Expected:**

```
Amine    13.67
Sara     18.0
Omar     10.33
Zoé      15.33
best -> Sara
```

In [ ]:
import json

# TODO

## Exercise 2 - Build a NEW object and write it

Build this structure **from scratch** in Python (do not modify the loaded one):

```python
report = {
    "class": "L3-Info",
    "count": 4,
    "results": [ {"name": ..., "average": ...}, ... ],   # sorted by average, best first
    "best": "Sara"
}
```

Then write it to `../data/report.json` with `indent=2`, and open the file in
PyCharm to check it looks right.

In [ ]:
# TODO

## Exercise 3 - `ensure_ascii` - the accent trap

Run both:

```python
print(json.dumps({"name": "Zoé"}))
print(json.dumps({"name": "Zoé"}, ensure_ascii=False))
```

**Expected:**

```
{"name": "Zo\u00e9"}
{"name": "Zoé"}
```

By default Python escapes every non-ASCII character. It is still *correct* JSON -
any parser reads `\u00e9` back as `é` - but it is unreadable for a human.

**Rule of thumb:** writing a file a person will open ->
`json.dump(obj, f, indent=2, ensure_ascii=False)`, and always
`encoding="utf-8"` on the `open`.

Re-dump your `report.json` with `ensure_ascii=False` and compare the two files.

In [ ]:
# TODO

## Exercise 4 - What JSON cannot store

JSON has 6 types only (object, array, string, number, bool, null). Python has
many more. Try each of these:

```python
json.dumps({"point": (1, 2)})        # tuple
json.dumps({"tags": {"a", "b"}})     # set
import datetime
json.dumps({"when": datetime.date(2026, 7, 23)})
```

**What happens:**
- tuple -> works, becomes an array: `{"point": [1, 2]}`.
  Careful: it comes back as a **list**, not a tuple. The round-trip is lossy.
- set -> `TypeError: Object of type set is not JSON serializable`
- date -> `TypeError: Object of type date is not JSON serializable`

Fix them yourself: convert before dumping (`list(my_set)`, `d.isoformat()`), or
pass `default=str` and let Python stringify whatever it does not know.

Try `json.dumps({"when": datetime.date(2026,7,23)}, default=str)` and look at the
result. Then ask yourself: is `default=str` a good idea for the *set* case?

In [ ]:
# TODO

## Exercise 5 - Keys are always strings

Run:

```python
s = json.dumps({1: "a", 2: "b"})
print(s)
print(json.loads(s))
```

**Expected:**

```
{"1": "a", "2": "b"}
{'1': 'a', '2': 'b'}
```

Your int keys came back as **strings**. JSON object keys are strings, full stop -
so `dumps` silently converts, and `loads` cannot know they were ints.

Why you should care: the org chart index `{ id: person }` with int ids will not
survive a JSON round-trip unchanged. If ids matter, keep them **inside** the
record (`{"id": 1, ...}`) and rebuild the index after loading - which is exactly
what `company.json` does.

In [ ]:
# TODO

## Recap - 04

- `json.dump(obj, f, indent=2, ensure_ascii=False)` is the good default for a human file.
- `ensure_ascii=False` + `encoding="utf-8"` keeps `é` readable.
- tuple -> array (lossy), set / date -> `TypeError`. Convert first, or `default=str`.
- Object keys are **always** strings after a round-trip.